# LC 91 — Decode Ways
**Day-66 | Theme: 1D Dynamic Programming | Difficulty: Medium**

<div style="border-left:4px solid purple; padding:10px 16px;
background:#f5f0ff; margin-top:12px">

**Core Insight:** At each position, count how many valid decodings
end here by asking two questions: can the *last one digit* decode
alone (non-zero), and can the *last two digits* decode as a valid
letter (10–26)? Add the corresponding DP values for each yes.

</div>

## Official Problem Statement

A message containing letters `A-Z` can be **encoded** into numbers
using the mapping `'A' -> "1", 'B' -> "2", ..., 'Z' -> "26"`.

To **decode** an encoded message, all digits must be mapped back into
letters using the reverse of the mapping above (there may be multiple
ways). Given a string `s` containing only digits, return the **number
of ways** to decode it.

The test cases are generated so that the answer fits in a **32-bit**
integer.

**Constraints:**
- `1 <= s.length <= 100`
- `s` contains only digits
- `s` may contain leading zeros

## What This Is Actually Asking

You have a string of digits. Each digit or pair of digits may
correspond to a letter (1–9 single, 10–26 pair). Count every
distinct split of the string that produces a valid all-letter
message.

A `'0'` on its own has no mapping, so it can never stand alone; it
must always be the second digit of `10` or `20`.

Any other leading zero immediately makes that path invalid, trimming
the count.

`dp[i]` stores the number of valid decodings for the first `i`
characters, so `dp[n]` is the final answer.

## Walk Through an Example by Hand

**Input:** `s = "226"`  — expected: `3`

```
Mappings: 2->B, 22->V, 26->Z, 6->F
Valid decodings: "BBF", "VF", "BZ"

dp array of length n+1=4, dp[0]=1 (empty string, base case)

i=1, s[0]='2' (not '0'):
  1-char: s[0]='2' != '0'  -> dp[1] += dp[0] = 1
  dp[1] = 1

i=2, s[1]='2' (not '0'):
  1-char: s[1]='2' != '0'  -> dp[2] += dp[1] = 1
  2-char: s[0:2]='22', 10<=22<=26 -> dp[2] += dp[0] = 1
  dp[2] = 2

i=3, s[2]='6' (not '0'):
  1-char: s[2]='6' != '0'  -> dp[3] += dp[2] = 2
  2-char: s[1:3]='26', 10<=26<=26 -> dp[3] += dp[1] = 1
  dp[3] = 3

Answer: dp[3] = 3
```

## The Picture

**Input:** `s = "12"` — expected: `2`

```
 index:  0    1    2
         |    |    |
 dp   : [1]  [1]  [2]
         ^         ^
      base case   dp[2] = dp[1] (use '2' alone)
                        + dp[0] (use '12' as pair)

Decision tree for "12":

  "12"
  /          \
"1"          "12"
  \            |
  "2"         'L'
   |
  'B'

 Path 1: 1->A  2->B   => "AB"
 Path 2: 12->L        => "L"

dp fills left to right; each cell sums its one-back
and two-back contributions.
```

## When To Use This Pattern

- When you must **count distinct ways** to parse or split a string,
  think 1D DP where `dp[i]` = ways to handle first `i` chars.
- When each position has **1 or 2 character** options that must be
  validated, think: check last-one and last-two separately.
- When a **'0' invalidates** a path, think: guard with
  `if s[i-1] != '0'` before adding `dp[i-1]`.
- When the alphabet is bounded (here 1–26), think: hard-code the
  range check `10 <= int(two_chars) <= 26`.
- When you see "number of ways to decode/parse a digit string",
  think LC 91 pattern immediately.

## The Approach

Build a DP array of length `n+1` where `dp[0]=1` (the empty prefix
has exactly one decoding: do nothing).

For each index `i` from 1 to `n`, add `dp[i-1]` to `dp[i]` if the
single digit `s[i-1]` is not `'0'` (it maps to letters A–I or A–Z
individually). Also add `dp[i-2]` (if `i>=2`) when the two-digit
number `s[i-2:i]` falls in the range `[10, 26]`.

If `dp[i]` remains 0 mid-way (e.g., a bare `'0'`), there are no
valid decodings from that point onward, and the final answer will
also be 0.

Return `dp[n]`.

In [ ]:
from typing import List

In [ ]:
def test_harness(func):
    """
    Run test cases for Decode Ways.
    Each tuple: (s, expected_output)
    """
    cases = [
        # (s,       expected)
        ("12",      2),  # AB or L
        ("226",     3),  # BBF, VF, BZ
        ("06",      0),  # leading 0 -> invalid
        ("10",      1),  # J only (0 can't stand alone)
        ("27",      1),  # BG only (27 > 26)
        ("11106",   2),  # AAJF, KJF
        ("0",       0),  # bare zero
        ("1",       1),  # A
    ]
    passed = 0
    for s, expected in cases:
        result = func(s)
        status = "PASSED" if result == expected else "FAILED"
        if status == "PASSED":
            passed += 1
        else:
            print(
                f"  {status} | s={s!r}"
                f" | expected={expected} | got={result}"
            )
    print(
        f"\nSummary: {passed}/{len(cases)} tests passed."
    )

# test_harness(num_decodings)  # uncomment once defined

In [ ]:
def num_decodings(s: str) -> int:
    """
    Return the number of ways to decode the digit string s.

    Strategy:
      - dp[i] = number of valid decodings of s[:i]
      - dp[0] = 1 (empty prefix, base case)
      - dp[1] = 0 if s[0]=='0' else 1
      - For i in 2..n:
          if s[i-1] != '0':  dp[i] += dp[i-1]
          two = int(s[i-2:i])
          if 10 <= two <= 26: dp[i] += dp[i-2]
      - Return dp[n]

    Args:
        s: non-empty string of digits

    Returns:
        Count of valid alphabetic decodings.

    Examples:
        >>> num_decodings("12")
        2
        >>> num_decodings("06")
        0
    """
    # Debug: inspect input
    print(f"[DEBUG] s={s!r}, n={len(s)}")

    # Debug: print dp array after building
    # print(f"[DEBUG] dp={dp}")

    # Debug: trace each iteration
    # print(f"[DEBUG] i={i} one_char={s[i-1]!r} "
    #       f"two_char={s[i-2:i]!r} dp[i]={dp[i]}")

    # Debug: final result
    # print(f"[DEBUG] answer=dp[{len(s)}]={dp[-1]}")

    pass

In [ ]:
# Uncomment and run when solution is ready
# test_harness(num_decodings)

## Complexity

| Approach | Time | Space | Notes |
|---|---|---|---|
| Brute force (recursion, no memo) | O(2^n) | O(n) stack | Recomputes subproblems |
| Memoized recursion (top-down DP) | O(n) | O(n) | Cache per index |
| Optimal (bottom-up DP) | O(n) | O(n) | dp array, clean |
| Space-optimised (two vars) | O(n) | O(1) | Only prev two values needed |

## Real World Connection

At **Citi**, SWIFT payment message parsers must count valid field
segmentations when a numeric tag sequence can map to either a
one-field or two-field code — the exact same one-back / two-back
DP logic.

On **AWS**, Kinesis stream processors that decode variable-length
binary-encoded events use a similar counting DP to validate how
many legal frame boundaries exist in a raw byte buffer before
committing an offset.

In **data engineering**, ETL pipelines that ingest compressed
category codes (e.g., product SKU strings) often need to count
how many valid decode paths exist before choosing the canonical
one, ensuring data quality by flagging strings with zero valid
decodings early in the pipeline.

> **Simplicity and clarity is Gold.** — Sean's Study Mantra